# GeoWatch AI — DOTA4 конкурсное обучение
Воспроизводимый YOLOv8-OBB baseline по aircraft, ship, small vehicle и large vehicle.

Ноутбук загружает Ultralytics-архив DOTAv1 (~2 GB), который ссылается на официальный DOTA v1.0. Набор разрешён только для академического использования; не используйте его в коммерческом продукте.

Перед запуском выберите Runtime → Change runtime type → T4 GPU. Первая ячейка остановится на CPU, чтобы случайно не начать многодневное обучение. После смены runtime запускайте ячейки сверху вниз.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU не подключена. Выберите Runtime → Change runtime type → T4 GPU, затем запустите ноутбук заново.'
!nvidia-smi
!pip -q install 'ultralytics>=8.3,<9' pyyaml
# Артефакты хранятся только в текущей Colab-сессии. Финальная ячейка скачивает веса и метрики.

In [ ]:
from pathlib import Path
import zipfile, urllib.request
ARCHIVE=Path('/content/DOTAv1.zip')
URL='https://github.com/ultralytics/assets/releases/download/v0.0.0/DOTAv1.zip'
if not ARCHIVE.exists(): urllib.request.urlretrieve(URL, ARCHIVE)
if not any((p/'images/train').exists() for p in Path('/content').glob('**/DOTAv1')):
    with zipfile.ZipFile(ARCHIVE) as z: z.extractall('/content')
SOURCE=next(p for p in Path('/content').glob('**/DOTAv1') if (p/'images/train').exists() and (p/'labels/train').exists())
assert (SOURCE/'images/val').exists() and (SOURCE/'labels/val').exists()
print('Официальный источник подготовлен:', SOURCE)

In [ ]:
# Строим DOTA4. DOTA IDs: plane=0, ship=1, large-vehicle=9, small-vehicle=10
import random, shutil, json, hashlib
from collections import Counter
REMAP={0:0,1:1,9:3,10:2}; NAMES={0:'aircraft',1:'ship',2:'small vehicle',3:'large vehicle'}
OUT=Path('/content/dota4'); rng=random.Random(42)
if OUT.exists(): shutil.rmtree(OUT)
def images(folder): return sorted(Path(folder).glob('*'))
def scene(p): return p.stem.split('__',1)[0]
train=images(SOURCE/'images/train'); val=images(SOURCE/'images/val')
scenes=sorted({scene(p) for p in train}); rng.shuffle(scenes); test_scenes=set(scenes[:max(1,round(len(scenes)*.2))])
splits={'train':[p for p in train if scene(p) not in test_scenes],'test':[p for p in train if scene(p) in test_scenes],'val':val}
counts={}; clipped={}
for split, files in splits.items():
  counts[split]=Counter(); clipped[split]=0
  for image in files:
    target=OUT/'images'/split/image.name; target.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(image,target)
    src_label=SOURCE/'labels'/('val' if split=='val' else 'train')/(image.stem+'.txt'); dst=OUT/'labels'/split/(image.stem+'.txt'); dst.parent.mkdir(parents=True,exist_ok=True); kept=[]
    for line in src_label.read_text().splitlines():
      row=line.split(); old=int(float(row[0]))
      if old not in REMAP: continue
      xy=[float(v) for v in row[1:]]; assert len(xy)==8
      if any(v<0 or v>1 for v in xy): clipped[split]+=1; xy=[min(1,max(0,v)) for v in xy]
      kept.append(str(REMAP[old])+' '+' '.join(f'{v:.8f}' for v in xy)); counts[split][REMAP[old]]+=1
    dst.write_text('\n'.join(kept)+'\n' if kept else '')
assert not ({scene(p) for p in splits['train']} & {scene(p) for p in splits['test']})
yaml={'path':str(OUT),'train':'images/train','val':'images/val','test':'images/test','names':NAMES}
(OUT/'dota4.yaml').write_text(json.dumps(yaml,indent=2))
fingerprint=hashlib.sha256()
for split in ('train','val','test'):
  for label in sorted((OUT/'labels'/split).glob('*.txt')):
    fingerprint.update(f'{split}/{label.name}\0'.encode()); fingerprint.update(label.read_bytes())
manifest={'seed':42,'split':'scene-separated','images':{k:len(v) for k,v in splits.items()},'instances':{k:{NAMES[c]:n for c,n in x.items()} for k,x in counts.items()},'edge_clipped_polygons':clipped,'dataset_fingerprint':'sha256:'+fingerprint.hexdigest()}
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2)); print(json.dumps(manifest,indent=2))

In [ ]:
# Gate: все 4 класса должны присутствовать в изолированном test.
required=set(NAMES.values()); test_classes={NAMES[k] for k in counts['test']}
missing=required-test_classes
assert not missing, f'Недостаточно классов в test: {sorted(missing)}. Измените seed или увеличьте source dataset.'
print('Dataset gate passed')

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU runtime обязателен для этого обучения.'
from google.colab import drive
drive.mount('/content/drive')
from ultralytics import YOLO
RUNS=Path('/content/drive/MyDrive/GeoWatch_AI/runs'); RUNS.mkdir(parents=True,exist_ok=True)
RUN_DIR=RUNS/'dota4_seed42'; RUN_DIR.mkdir(parents=True,exist_ok=True)
shutil.copy2(OUT/'manifest.json', RUN_DIR/'dataset_manifest.json')
model=YOLO('yolov8n-obb.pt')
model.train(data=str(OUT/'dota4.yaml'),epochs=80,imgsz=1024,batch=4,device=0,workers=2,seed=42,deterministic=True,patience=15,close_mosaic=10,max_det=2000,save_period=5,project=str(RUNS),name='dota4_seed42',exist_ok=True)

In [ ]:
import json
best=RUNS/'dota4_seed42/weights/best.pt'
assert best.exists(), 'best.pt не найден: сначала завершите ячейку model.train(...).'
trained=YOLO(str(best))
m=trained.val(data=str(OUT/'dota4.yaml'),split='test',imgsz=1024,device=0,conf=.001,plots=True)
p,r=float(m.box.mp),float(m.box.mr); checkpoint_sha256=hashlib.sha256(best.read_bytes()).hexdigest()
report={'dataset':'DOTA4 scene-separated internal test','split':'test','dataset_fingerprint':manifest['dataset_fingerprint'],'checkpoint_sha256':checkpoint_sha256,'model':'yolov8n-obb','seed':42,'epochs_requested':80,'threshold_for_eval':.001,'precision':p,'recall':r,'f1':2*p*r/max(p+r,1e-12),'map50':float(m.box.map50),'map50_95':float(m.box.map),'note':'Selected on validation and evaluated once on isolated internal test.'}
(RUNS/'dota4_seed42/metrics_test.json').write_text(json.dumps(report,indent=2)); print(json.dumps(report,indent=2))
from google.colab import files
files.download(str(best)); files.download(str(RUNS/'dota4_seed42/metrics_test.json'))